# MaskGen Token Regret Critic (Single-Notebook DDP Training)


## Setup: Import and Load

In [ ]:
import os
import sys
import importlib
import torch
import open_clip
import matplotlib.pyplot as plt
from modeling.tatitok import TATiTok
from modeling.maskgen import MaskGen_VQ

# Resolve the repo root robustly so notebook imports and local checkpoints stay aligned.
PROJECT_ROOT_CANDIDATES = [
    os.path.abspath('.'),
    os.path.dirname(os.path.abspath('.')),
    os.path.dirname(os.path.dirname(os.path.abspath('.'))),
]
PROJECT_ROOT = next(
    (path for path in PROJECT_ROOT_CANDIDATES if os.path.isdir(os.path.join(path, 'token_regrate'))),
    os.path.abspath('.'),
)
HF_CACHE_DIR = os.path.abspath(os.environ.get('HF_CACHE_DIR', os.path.join(PROJECT_ROOT, 'hf_cache')))
LOCAL_FILES_ONLY = bool(HF_CACHE_DIR)
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = HF_CACHE_DIR
os.environ['OPENCLIP_CACHE_DIR'] = HF_CACHE_DIR
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print('Project root:', PROJECT_ROOT)
print('HF cache:', HF_CACHE_DIR)
print('LOCAL_FILES_ONLY:', LOCAL_FILES_ONLY)

# Add project root to path for imports and make relative checkpoints/data paths stable.
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)


import token_regrate.config as token_regret_config
import token_regrate.train_token_regret_ddp as token_regret_train
token_regret_config = importlib.reload(token_regret_config)
token_regret_train = importlib.reload(token_regret_train)
get_config = token_regret_config.get_config

# Import from the reloaded training script - shared training and inference utilities.
from token_regrate.train_token_regret_ddp import *

def _resolve_project_path(path_value):
    path_value = os.path.expanduser(str(path_value))
    return path_value if os.path.isabs(path_value) else os.path.abspath(os.path.join(PROJECT_ROOT, path_value))

def _resolve_pretrained_path(path_value, cache_root=HF_CACHE_DIR):
    path_value = os.path.expanduser(str(path_value))
    if os.path.isabs(path_value):
        return path_value

    normalized_path = os.path.normpath(path_value)
    cache_relative_path = normalized_path
    cache_prefix = f'hf_cache{os.sep}'
    if normalized_path.startswith(cache_prefix):
        cache_relative_path = normalized_path.split(os.sep, 1)[1]

    candidates = [
        os.path.abspath(normalized_path),
        os.path.abspath(os.path.join(PROJECT_ROOT, normalized_path)),
        os.path.abspath(os.path.join(cache_root, cache_relative_path)),
    ]
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return candidates[0]

def load_pretrained_stack(model_cfg=None):
    model_cfg = get_config().model if model_cfg is None else model_cfg
    tok_dir = _resolve_pretrained_path(model_cfg.tokenizer_path)
    gen_dir = _resolve_pretrained_path(model_cfg.generator_path)
    clip_name = str(model_cfg.clip_name)
    clip_pretrained = str(getattr(model_cfg, 'clip_pretrained', 'openai'))
    clip_force_quick_gelu = bool(getattr(model_cfg, 'clip_force_quick_gelu', True))

    print('Loading TA-TiTok tokenizer...')
    tatitok_vq_tokenizer = TATiTok.from_pretrained(pretrained_model_name_or_path=tok_dir, cache_dir=tok_dir)
    tatitok_vq_tokenizer.eval()
    tatitok_vq_tokenizer.requires_grad_(False)

    print('Loading MaskGen-VQ generator...')
    maskgen_vq_generator = MaskGen_VQ.from_pretrained(pretrained_model_name_or_path=gen_dir, cache_dir=gen_dir)
    maskgen_vq_generator.eval()
    maskgen_vq_generator.requires_grad_(False)

    print('Loading CLIP text encoder...')
    clip_encoder, _, _ = open_clip.create_model_and_transforms(
        clip_name, pretrained=clip_pretrained, force_quick_gelu=clip_force_quick_gelu
    )
    del clip_encoder.visual
    clip_tokenizer = open_clip.get_tokenizer(clip_name)
    clip_encoder.transformer.batch_first = False
    clip_encoder.eval()
    clip_encoder.requires_grad_(False)

    print('Pretrained stack ready.')
    return tatitok_vq_tokenizer, maskgen_vq_generator, clip_tokenizer, clip_encoder


tatitok_vq_tokenizer, maskgen_vq_generator, clip_tokenizer, clip_encoder = load_pretrained_stack()
maskgen_vq_generator = maskgen_vq_generator.to(device)
tatitok_vq_tokenizer = tatitok_vq_tokenizer.to(device)
clip_encoder = clip_encoder.to(device)
print('Generator device:', next(maskgen_vq_generator.parameters()).device)
print('Tokenizer device:', next(tatitok_vq_tokenizer.parameters()).device)
print('CLIP text encoder device:', next(clip_encoder.parameters()).device)


## Init all parameters for training and inference time

In [ ]:
# Load notebook defaults directly from token_regrate/config.py.
CONFIG = get_config()
INF_CFG = CONFIG.inference
TRAIN_CFG = CONFIG.training
DATA_CFG = CONFIG.dataset
EXPERIMENT_CFG = CONFIG.experiment
RUNTIME_CFG = CONFIG.runtime
MODEL = CONFIG.model
NOTEBOOK_GENERATE_KWARGS = {
    'guidance_scale': float(TRAIN_CFG.train_guidance_scale),
    'randomize_temperature': float(TRAIN_CFG.train_randomize_temperature),
    'aesthetic_score': float(TRAIN_CFG.train_aesthetic_score),
    'num_sample_steps': int(MODEL.sample_steps),
    'remask_ratio': float(INF_CFG.remask_ratio),
    'refine_loops': int(INF_CFG.refine_loops),
    'refine_start_step': int(INF_CFG.refine_start_step),
    'repair_greedy': bool(INF_CFG.repair_greedy),
}

print('Notebook config loaded from token_regrate/config.py')
print('Inference steps:', MODEL.sample_steps)
print('Training epochs:', TRAIN_CFG.num_epochs)
print('Training seed:', TRAIN_CFG.seed)
print('Training batch size:', TRAIN_CFG.per_gpu_batch_size)
print('Max training images:', getattr(TRAIN_CFG, 'max_train_images', 0))
print('Training output dir:', EXPERIMENT_CFG.output_dir)
print('Learning rate:', TRAIN_CFG.learning_rate)
print('Counterfactual rollout steps:', TRAIN_CFG.counterfactual_rollout_steps)
print('Counterfactual utility:', getattr(TRAIN_CFG, 'counterfactual_utility', 'token_ce'))
print('Counterfactual window radius:', getattr(TRAIN_CFG, 'counterfactual_window_radius', 0))
print('Counterfactual contrast negatives:', getattr(TRAIN_CFG, 'counterfactual_contrast_negatives', 2))
print('Counterfactual contrast temperature:', getattr(TRAIN_CFG, 'counterfactual_contrast_temperature', 1.0))
print('Counterfactual contrast mode:', getattr(TRAIN_CFG, 'counterfactual_contrast_mode', 'nce'))
print('Counterfactual repair greedy:', getattr(TRAIN_CFG, 'counterfactual_repair_greedy', None))
print('Critic prompt gap top-k:', getattr(TRAIN_CFG, 'critic_prompt_gap_topk', 0))
print('Regret target transform:', getattr(TRAIN_CFG, 'regret_target_transform', None))
print('Inference selection: pure top-k (no score threshold)')
print('Train repair greedy:', getattr(TRAIN_CFG, 'train_repair_greedy', None))
print('Fixed rollout step:', getattr(TRAIN_CFG, 'fixed_rollout_step', None))
print('Use target replay:', getattr(TRAIN_CFG, 'use_target_critic_replay', None))
print('Lambda rank:', TRAIN_CFG.lambda_rank)
print('Resume checkpoint:', repr(str(RUNTIME_CFG.resume_checkpoint)))
print('DDP only:', bool(TRAIN_CFG.ddp_only))


## Critic Training Entry Cell

Official mode: train only the critic head from the provided dataset source.


In [ ]:
import os, torch
print("cuda_available:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
print("TOKEN_REGRET_NPROC:", os.environ.get("TOKEN_REGRET_NPROC"))
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


In [ ]:
import subprocess
trained_critic = None

token_regret_nproc = os.environ.get('TOKEN_REGRET_NPROC')
if not torch.cuda.is_available():
    nproc_per_node = 1
else:
    visible_gpu_count = torch.cuda.device_count()
    # NCCL fails during communicator allocation on this host; use two ranks with Gloo by default.
    requested_nproc = int(token_regret_nproc) if token_regret_nproc else min(2, visible_gpu_count)
    if requested_nproc > visible_gpu_count:
        print(f'Reducing TOKEN_REGRET_NPROC={requested_nproc} to visible_gpu_count={visible_gpu_count}.')
    if token_regret_nproc is None and visible_gpu_count > 1:
        print(f'Defaulting TOKEN_REGRET_NPROC to {requested_nproc} with Gloo backend because multi-GPU NCCL init fails on this host.')
    nproc_per_node = max(1, min(requested_nproc, visible_gpu_count))
# if str(getattr(TRAIN_CFG, 'counterfactual_utility', '')).endswith('_contrast'):
#     max_train_images_for_contrast = int(getattr(TRAIN_CFG, 'max_train_images', 0))
#     if max_train_images_for_contrast > 0:
#         max_safe_nproc = max(1, max_train_images_for_contrast // 2)
#         if nproc_per_node > max_safe_nproc:
#             print(f'Contrastive target needs at least two prompts per rank; reducing nproc_per_node from {nproc_per_node} to {max_safe_nproc}.')
#             nproc_per_node = max_safe_nproc
#         if (max_train_images_for_contrast + nproc_per_node - 1) // nproc_per_node < 2:
#             raise RuntimeError('Contrastive counterfactual training needs at least two prompts per local rank.')
ddp_cmd = [
    sys.executable,
    '-m', 'torch.distributed.run',
    f'--nproc_per_node={nproc_per_node}',
    'token_regrate/train_token_regret_ddp.py',
    '--num-epochs', str(TRAIN_CFG.num_epochs),
    '--max-train-images', str(getattr(TRAIN_CFG, 'max_train_images', 0)),
    '--per-gpu-batch-size', str(TRAIN_CFG.per_gpu_batch_size),
    '--learning-rate', str(TRAIN_CFG.learning_rate),
    '--counterfactual-rollout-steps', str(TRAIN_CFG.counterfactual_rollout_steps),
    '--counterfactual-utility', str(getattr(TRAIN_CFG, 'counterfactual_utility', 'token_ce')),
    '--counterfactual-window-radius', str(getattr(TRAIN_CFG, 'counterfactual_window_radius', 0)),
    '--counterfactual-contrast-negatives', str(getattr(TRAIN_CFG, 'counterfactual_contrast_negatives', 2)),
    '--counterfactual-contrast-temperature', str(getattr(TRAIN_CFG, 'counterfactual_contrast_temperature', 1.0)),
    '--counterfactual-contrast-mode', str(getattr(TRAIN_CFG, 'counterfactual_contrast_mode', 'nce')),
    '--critic-prompt-gap-topk', str(getattr(TRAIN_CFG, 'critic_prompt_gap_topk', 8)),
    '--regret-target-transform', str(getattr(TRAIN_CFG, 'regret_target_transform', 'tanh')),
    '--target-critic-ema-decay', str(TRAIN_CFG.target_critic_ema_decay),
    '--fixed-rollout-step', str(getattr(TRAIN_CFG, 'fixed_rollout_step', -1)),
    '--lambda-rank', str(TRAIN_CFG.lambda_rank),
    '--rank-margin', str(TRAIN_CFG.rank_margin),
    '--rank-gap-threshold', str(TRAIN_CFG.rank_gap_threshold),
    '--train-remask-ratio', str(TRAIN_CFG.train_remask_ratio),
    '--train-refine-start-step', str(TRAIN_CFG.train_refine_start_step),
    '--counterfactual-chunk-size', str(TRAIN_CFG.counterfactual_chunk_size),
    '--refine-loops', str(TRAIN_CFG.refine_loops),
    '--save-every', str(TRAIN_CFG.save_every),
    '--log-every', str(TRAIN_CFG.log_every),
    '--seed', str(TRAIN_CFG.seed),
    '--output-dir', str(EXPERIMENT_CFG.output_dir),
    '--train-data-source', str(DATA_CFG.source),
    '--train-dataset-mode', str(DATA_CFG.mode),
    '--cc12m-cache-dir', str(DATA_CFG.cc12m_cache_dir),
    '--stream-prefetch-batches', str(DATA_CFG.stream_prefetch_batches),
    '--cc12m-loader-workers', str(DATA_CFG.cc12m_loader_workers),
    '--cc12m-loader-max-pending', str(DATA_CFG.cc12m_loader_max_pending),
]
if int(getattr(TRAIN_CFG, 'max_steps', 0)) > 0:
    ddp_cmd += ['--max-steps', str(TRAIN_CFG.max_steps)]
counterfactual_repair_greedy = bool(getattr(TRAIN_CFG, 'counterfactual_repair_greedy', getattr(INF_CFG, 'repair_greedy', False)))
train_repair_greedy = bool(getattr(TRAIN_CFG, 'train_repair_greedy', getattr(TRAIN_CFG, 'counterfactual_repair_greedy', getattr(INF_CFG, 'repair_greedy', False))))
if counterfactual_repair_greedy:
    ddp_cmd += ['--counterfactual-repair-greedy']
else:
    ddp_cmd += ['--no-counterfactual-repair-greedy']
if train_repair_greedy:
    ddp_cmd += ['--train-repair-greedy']
else:
    ddp_cmd += ['--no-train-repair-greedy']
if bool(DATA_CFG.disable_cc12m_cache):
    ddp_cmd += ['--disable-cc12m-cache']
resume_checkpoint = str(getattr(RUNTIME_CFG, 'resume_checkpoint', '')).strip()
if resume_checkpoint:
    ddp_cmd += ['--resume-checkpoint', resume_checkpoint]

ddp_env = os.environ.copy()
dist_backend = os.environ.get('TOKEN_REGRET_DIST_BACKEND') or ('gloo' if nproc_per_node > 1 else str(TRAIN_CFG.dist_backend))
ddp_env['DIST_BACKEND'] = dist_backend
# Keep NCCL workaround flags available when TOKEN_REGRET_DIST_BACKEND=nccl is explicitly requested.
ddp_env['NCCL_DEBUG'] = 'INFO'
ddp_env['NCCL_P2P_DISABLE'] = '1'
ddp_env['NCCL_IB_DISABLE'] = '1'
ddp_env['TORCH_NCCL_ASYNC_ERROR_HANDLING'] = '1'
ddp_env['NCCL_CUMEM_ENABLE'] = '0'
ddp_env['NCCL_CUMEM_HOST_ENABLE'] = '0'
print('Launching DDP training:', ' '.join(ddp_cmd))
print('DDP environment:', {k: ddp_env.get(k) for k in ['DIST_BACKEND', 'NCCL_P2P_DISABLE', 'NCCL_IB_DISABLE', 'NCCL_DEBUG', 'NCCL_CUMEM_ENABLE', 'NCCL_CUMEM_HOST_ENABLE', 'TORCH_NCCL_ASYNC_ERROR_HANDLING']})
run = subprocess.run(ddp_cmd, check=False, env=ddp_env)
if run.returncode != 0:
    raise RuntimeError(f'DDP training failed with return code {run.returncode}')

## This cell works only for the subset of data

In [ ]:
# Generate images and compare them to the saved ground-truth training subset.
# Columns: GT image, baseline MaskGen generation, critic-guided generation.
import json
import os
import textwrap
from PIL import Image
import matplotlib.pyplot as plt

from token_regrate.train_token_regret_ddp import generate_image_vq_batch, set_global_seed
from token_regrate.utils import load_trained_critic

if 'CONFIG' not in globals():
    CONFIG = get_config()
INF_CFG = globals().get('INF_CFG', CONFIG.inference)
TRAIN_CFG = globals().get('TRAIN_CFG', CONFIG.training)
EXPERIMENT_CFG = globals().get('EXPERIMENT_CFG', CONFIG.experiment)
RUNTIME_CFG = globals().get('RUNTIME_CFG', CONFIG.runtime)
CRITIC_PROMPT_GAP_TOPK = int(getattr(TRAIN_CFG, 'critic_prompt_gap_topk', 8))
MODEL = globals().get('MODEL', CONFIG.model)
VIZ_CRITIC_PROMPT_GAP_TOPK = int(getattr(TRAIN_CFG, 'critic_prompt_gap_topk', 8))


def _viz_resolve_path(path_value):
    path_value = os.path.expanduser(str(path_value))
    if os.path.isabs(path_value):
        return path_value
    root = globals().get('PROJECT_ROOT', os.path.abspath('.'))
    return os.path.abspath(os.path.join(root, path_value))


def _load_viz_subset(manifest_path):
    manifest_path = _viz_resolve_path(manifest_path)
    if not os.path.isfile(manifest_path):
        raise FileNotFoundError(f'Used-training manifest not found: {manifest_path}')

    subset_root = os.path.dirname(manifest_path)
    rows, images, captions = [], [], []
    with open(manifest_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            image_path = str(row.get('image', '')).strip()
            if image_path and not os.path.isabs(image_path):
                image_path = os.path.join(subset_root, image_path)
            if not os.path.isfile(image_path):
                raise FileNotFoundError(f'Subset image missing: {image_path}')
            with Image.open(image_path) as img:
                images.append(img.convert('RGB'))
            row['image_path'] = image_path
            rows.append(row)
            captions.append(str(row.get('caption', '')).strip())
    if not captions:
        raise RuntimeError(f'No examples found in {manifest_path}')
    return rows, images, captions


def _resolve_viz_checkpoint():
    if '_resolve_critic_checkpoint' in globals():
        try:
            return _resolve_critic_checkpoint(globals().get('CRITIC_CKPT_PATH', str(getattr(RUNTIME_CFG, 'resume_checkpoint', ''))))
        except FileNotFoundError:
            pass
    candidates = [
        globals().get('CRITIC_CKPT_PATH', ''),
        os.environ.get('TOKEN_REGRET_VIZ_CKPT', '').strip(),
        str(getattr(RUNTIME_CFG, 'resume_checkpoint', '')).strip(),
        os.path.join(str(EXPERIMENT_CFG.output_dir), 'critic_last.pt'),
    ]
    for candidate in candidates:
        if candidate:
            candidate = _viz_resolve_path(candidate)
            if os.path.isfile(candidate):
                return candidate
    return None


def _generate_viz_images(prompts, use_critic_head=False, critic=None, seed_offset=0):
    images = []
    batch_size = max(1, int(VIZ_BATCH_SIZE))
    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]
        set_global_seed(int(VIZ_SEED) + int(seed_offset) + int(start))
        images.extend(
            generate_image_vq_batch(
                prompts=batch_prompts,
                model=maskgen_vq_generator,
                tokenizer=tatitok_vq_tokenizer,
                clip_tokenizer=clip_tokenizer,
                clip_encoder=clip_encoder,
                guidance_scale=float(TRAIN_CFG.train_guidance_scale),
                randomize_temperature=float(TRAIN_CFG.train_randomize_temperature),
                aesthetic_score=float(TRAIN_CFG.train_aesthetic_score),
                num_sample_steps=int(MODEL.sample_steps),
                use_regret_remask=bool(use_critic_head),
                critic=critic,
                remask_ratio=float(INF_CFG.remask_ratio),
                refine_start_step=int(INF_CFG.refine_start_step),
                refine_loops=int(INF_CFG.refine_loops),
                repair_greedy=bool(INF_CFG.repair_greedy),
            )
        )
    return images


VIZ_MANIFEST_PATH = _viz_resolve_path(
    os.path.join(str(EXPERIMENT_CFG.output_dir), 'used_training_images', 'manifest.jsonl')
)
VIZ_START_INDEX = max(0, int(os.environ.get('TOKEN_REGRET_VIZ_START_INDEX', '0')))
VIZ_NUM_IMAGES = max(1, int(os.environ.get('TOKEN_REGRET_VIZ_NUM_IMAGES', '8')))
VIZ_BATCH_SIZE = max(1, int(os.environ.get('TOKEN_REGRET_VIZ_BATCH_SIZE', '4')))
VIZ_SEED = int(os.environ.get('TOKEN_REGRET_VIZ_SEED', int(INF_CFG.seed)))
VIZ_SHOW_BASELINE = os.environ.get('TOKEN_REGRET_VIZ_SHOW_BASELINE', '1').lower() in {'1', 'true', 'yes', 'on'}
VIZ_SHOW_CRITIC = os.environ.get('TOKEN_REGRET_VIZ_SHOW_CRITIC', '1').lower() in {'1', 'true', 'yes', 'on'}

viz_rows_all, viz_gt_all, viz_captions_all = _load_viz_subset(VIZ_MANIFEST_PATH)
viz_end = min(len(viz_captions_all), VIZ_START_INDEX + VIZ_NUM_IMAGES)
viz_rows = viz_rows_all[VIZ_START_INDEX:viz_end]
viz_gt_images = viz_gt_all[VIZ_START_INDEX:viz_end]
viz_captions = viz_captions_all[VIZ_START_INDEX:viz_end]
if not viz_captions:
    raise RuntimeError(
        f'No images selected. start={VIZ_START_INDEX}, count={VIZ_NUM_IMAGES}, total={len(viz_captions_all)}'
    )

viz_device = next(maskgen_vq_generator.parameters()).device
maskgen_vq_generator = maskgen_vq_generator.to(viz_device).eval()
tatitok_vq_tokenizer = tatitok_vq_tokenizer.to(viz_device).eval()
clip_encoder = clip_encoder.to(viz_device).eval()

viz_critic = None
if VIZ_SHOW_CRITIC:
    viz_ckpt_path = _resolve_viz_checkpoint()
    if viz_ckpt_path is None:
        VIZ_SHOW_CRITIC = False
    elif 'critic_head' in globals() and int(getattr(critic_head, 'prompt_gap_topk', VIZ_CRITIC_PROMPT_GAP_TOPK)) == VIZ_CRITIC_PROMPT_GAP_TOPK:
        viz_critic = critic_head.to(viz_device).eval()
    else:
        viz_critic = load_trained_critic(
            viz_ckpt_path,
            maskgen_vq_generator,
            use_hidden=True,
            prompt_gap_topk=VIZ_CRITIC_PROMPT_GAP_TOPK,
        ).to(viz_device).eval()

baseline_images = _generate_viz_images(viz_captions, use_critic_head=False, critic=None) if VIZ_SHOW_BASELINE else []
critic_images = _generate_viz_images(viz_captions, use_critic_head=True, critic=viz_critic) if VIZ_SHOW_CRITIC else []

columns = [('GT', viz_gt_images)]
if VIZ_SHOW_BASELINE:
    columns.append(('Generated', baseline_images))
if VIZ_SHOW_CRITIC:
    columns.append(('Critic-guided', critic_images))

fig_width = 4.2 * len(columns)
fig_height = max(3.0, 3.6 * len(viz_captions))
fig, axes = plt.subplots(len(viz_captions), len(columns), figsize=(fig_width, fig_height), squeeze=False)

for row_idx, caption in enumerate(viz_captions):
    short_caption = textwrap.shorten(caption.replace('\n', ' '), width=92, placeholder='...')
    for col_idx, (title, images) in enumerate(columns):
        ax = axes[row_idx][col_idx]
        ax.imshow(images[row_idx])
        ax.axis('off')
        heading = title if row_idx == 0 else ''
        if col_idx == 0:
            ax.set_title(f'{heading}\n#{VIZ_START_INDEX + row_idx}: {short_caption}', fontsize=9, loc='left')
        else:
            ax.set_title(heading, fontsize=10)

plt.tight_layout()
VIZ_OUTPUT_PATH = os.path.join(os.path.dirname(VIZ_MANIFEST_PATH), 'generated_vs_gt.png')
fig.savefig(VIZ_OUTPUT_PATH, dpi=160, bbox_inches='tight')
plt.show()

VIZ_COMPARISON = {
    'rows': viz_rows,
    'captions': viz_captions,
    'gt_images': viz_gt_images,
    'baseline_images': baseline_images,
    'critic_images': critic_images,
    'output_path': VIZ_OUTPUT_PATH,
}


## Inspect Code

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# Run this cell after the DDP training cell finishes.
INSPECT_OUTPUT_DIR = os.path.abspath(str(EXPERIMENT_CFG.output_dir))
INSPECT_MANIFEST = os.path.join(INSPECT_OUTPUT_DIR, 'used_training_images')
INSPECT_ROOT = os.path.join(INSPECT_OUTPUT_DIR, 'inspection_all')
INSPECT_CHECKPOINT_PATTERN = os.environ.get('TOKEN_REGRET_INSPECT_PATTERN', 'critic_last.pt')
INSPECT_BATCH_SIZE = int(os.environ.get('TOKEN_REGRET_INSPECT_BATCH_SIZE', '8'))
INSPECT_MAX_EXAMPLES = int(os.environ.get('TOKEN_REGRET_INSPECT_MAX_EXAMPLES', '32'))
INSPECT_DEVICE = os.environ.get('TOKEN_REGRET_INSPECT_DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')
INSPECT_SEED = int(os.environ.get('TOKEN_REGRET_INSPECT_SEED', str(getattr(TRAIN_CFG, 'seed', 123))))


def run_token_regret_inspection(
    output_dir=INSPECT_OUTPUT_DIR,
    manifest=INSPECT_MANIFEST,
    inspect_root=INSPECT_ROOT,
    checkpoint_pattern=INSPECT_CHECKPOINT_PATTERN,
    batch_size=INSPECT_BATCH_SIZE,
    max_examples=INSPECT_MAX_EXAMPLES,
    device=INSPECT_DEVICE,
    seed=INSPECT_SEED,
    skip_existing=True,
    ema_target=False,
):
    output_dir = os.path.abspath(str(output_dir))
    manifest = os.path.abspath(str(manifest))
    inspect_root = os.path.abspath(str(inspect_root))
    inspect_script = Path('token_regrate/inspect_token_regret_critic.py')
    if not inspect_script.is_file():
        raise FileNotFoundError(f'Inspection script not found: {inspect_script.resolve()}')

    cmd = [
        sys.executable,
        str(inspect_script),
        'directory',
        output_dir,
        '--output-dir', output_dir,
        '--checkpoint-pattern', str(checkpoint_pattern),
        '--manifest', manifest,
        '--inspect-root', inspect_root,
        '--batch-size', str(int(batch_size)),
        '--max-examples', str(int(max_examples)),
        '--seed', str(int(seed)),
        '--device', str(device),
        '--counterfactual-rollout-steps', str(int(getattr(TRAIN_CFG, 'counterfactual_rollout_steps', 2))),
    ]
    if int(getattr(TRAIN_CFG, 'counterfactual_chunk_size', 0)) > 0:
        cmd += ['--counterfactual-chunk-size', str(int(TRAIN_CFG.counterfactual_chunk_size))]
    if skip_existing:
        cmd.append('--skip-existing')
    if ema_target:
        cmd.append('--ema-target')

    print('Launching token-regret inspection:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

    summary_path = os.path.join(inspect_root, 'directory_summary.json')
    with open(summary_path, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    print(f"Inspection complete: {summary_path}")
    for row in summary.get('summaries', []):
        ckpt = os.path.basename(row.get('checkpoint', ''))
        step = row.get('checkpoint_step')
        ratio = row.get('mse_vs_constant_ratio')
        top_ratio = row.get('top_vs_oracle_ratio')
        print(f"{ckpt:>22} step={step!s:>6} mse/const={ratio!s:>10} top/oracle={top_ratio!s:>10}")
    return summary


INSPECTION_SUMMARY = run_token_regret_inspection()


# Optional GenEval baseline-vs-critic comparison on a small balanced subset.
# Default: 4 prompts from each of the 6 GenEval task tags, with 4 samples per prompt.
GENEVAL_CHECKPOINT = os.environ.get(
    'TOKEN_REGRET_GENEVAL_CHECKPOINT',
    os.path.join(INSPECT_OUTPUT_DIR, 'critic_last.pt'),
)
GENEVAL_OUTPUT_DIR = os.environ.get('TOKEN_REGRET_GENEVAL_OUTPUT_DIR', INSPECT_OUTPUT_DIR)
GENEVAL_DIR = os.environ.get(
    'TOKEN_REGRET_GENEVAL_DIR',
    os.path.join(INSPECT_OUTPUT_DIR, 'geneval_inspection', 'critic_last_4_per_task'),
)
GENEVAL_PROMPTS_PER_TASK = int(os.environ.get('TOKEN_REGRET_GENEVAL_PROMPTS_PER_TASK', '8'))
GENEVAL_BATCH_SIZE = int(os.environ.get('TOKEN_REGRET_GENEVAL_BATCH_SIZE', '4'))
GENEVAL_EVAL_PYTHON = os.environ.get(
    'TOKEN_REGRET_GENEVAL_EVAL_PYTHON',
    '/home/behzad/anaconda3/envs/geneval/bin/python',
)
GENEVAL_DEVICE = os.environ.get('TOKEN_REGRET_GENEVAL_DEVICE', INSPECT_DEVICE)
GENEVAL_SKIP_EVAL = os.environ.get('TOKEN_REGRET_GENEVAL_SKIP_EVAL', '0').lower() in {'1', 'true', 'yes', 'on'}


def run_token_regret_geneval(
    output_dir=GENEVAL_OUTPUT_DIR,
    checkpoint=GENEVAL_CHECKPOINT,
    geneval_dir=GENEVAL_DIR,
    prompts_per_task=GENEVAL_PROMPTS_PER_TASK,
    batch_size=GENEVAL_BATCH_SIZE,
    device=GENEVAL_DEVICE,
    eval_python=GENEVAL_EVAL_PYTHON,
    skip_existing=True,
    skip_eval=GENEVAL_SKIP_EVAL,
):
    output_dir = os.path.abspath(str(output_dir))
    checkpoint = os.path.abspath(str(checkpoint))
    geneval_dir = os.path.abspath(str(geneval_dir))
    inspect_script = Path('token_regrate/inspect_token_regret_critic.py')
    if not inspect_script.is_file():
        raise FileNotFoundError(f'Inspection script not found: {inspect_script.resolve()}')
    if not os.path.isfile(checkpoint):
        raise FileNotFoundError(f'Critic checkpoint not found: {checkpoint}')

    cmd = [
        sys.executable,
        str(inspect_script),
        'geneval',
        '--output-dir', output_dir,
        '--checkpoint', checkpoint,
        '--geneval-dir', geneval_dir,
        '--batch-size', str(int(batch_size)),
        '--prompts-per-task', str(int(prompts_per_task)),
        '--device', str(device),
        '--eval-python', str(eval_python),
    ]
    if skip_existing:
        cmd.append('--skip-existing')
    if skip_eval:
        cmd.append('--skip-eval')

    print('Launching balanced GenEval inspection:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

    summary_path = os.path.join(geneval_dir, 'comparison_summary.json')
    with open(summary_path, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    print(f"GenEval comparison complete: {summary_path}")
    comparison = summary.get('comparison', {})
    if comparison:
        print(
            'overall:',
            comparison.get('baseline_overall_score'),
            '->',
            comparison.get('head_overall_score'),
            'delta=',
            comparison.get('overall_score_delta'),
        )
        for tag, row in comparison.get('by_tag', {}).items():
            print(f"{tag:>16}: {row.get('baseline')} -> {row.get('head')} delta={row.get('delta')}")
    return summary


def show_geneval_saved_grids(geneval_dir=GENEVAL_DIR, max_prompts=None):
    geneval_dir = os.path.abspath(str(geneval_dir))
    baseline_root = os.path.join(geneval_dir, 'images', 'baseline')
    head_root = os.path.join(geneval_dir, 'images', 'head')
    prompt_dirs = sorted(
        name for name in os.listdir(baseline_root)
        if name.isdigit() and os.path.isdir(os.path.join(baseline_root, name))
    )
    if max_prompts is None:
        max_prompts = int(os.environ.get('TOKEN_REGRET_GENEVAL_SHOW_PROMPTS', str(len(prompt_dirs))))
    prompt_dirs = prompt_dirs[:max(1, int(max_prompts))]
    if not prompt_dirs:
        raise RuntimeError(f'No generated GenEval prompt folders found in {baseline_root}')

    fig, axes = plt.subplots(len(prompt_dirs), 2, figsize=(10, max(3, 3.2 * len(prompt_dirs))), squeeze=False)
    for row_idx, prompt_id in enumerate(prompt_dirs):
        baseline_prompt_dir = os.path.join(baseline_root, prompt_id)
        head_prompt_dir = os.path.join(head_root, prompt_id)
        with open(os.path.join(baseline_prompt_dir, 'metadata.jsonl'), 'r', encoding='utf-8') as f:
            metadata = json.load(f)
        prompt = str(metadata.get('prompt', '')).strip()
        tag = str(metadata.get('tag', '')).strip()
        for col_idx, (label, prompt_dir) in enumerate([('Baseline', baseline_prompt_dir), ('Critic-guided', head_prompt_dir)]):
            grid_path = os.path.join(prompt_dir, 'grid.png')
            if not os.path.isfile(grid_path):
                samples_dir = os.path.join(prompt_dir, 'samples')
                first_sample = sorted(x for x in os.listdir(samples_dir) if x.endswith('.png'))[0]
                grid_path = os.path.join(samples_dir, first_sample)
            ax = axes[row_idx][col_idx]
            ax.imshow(Image.open(grid_path).convert('RGB'))
            ax.axis('off')
            title = label if row_idx == 0 else ''
            if col_idx == 0:
                ax.set_title(f'{title}\n{prompt_id} [{tag}] {prompt}', fontsize=8, loc='left')
            else:
                ax.set_title(title, fontsize=9)
    plt.tight_layout()
    display_path = os.path.join(geneval_dir, 'balanced_geneval_grids.png')
    fig.savefig(display_path, dpi=140, bbox_inches='tight')
    plt.show()
    print(f'Saved displayed grid overview: {display_path}')
    return display_path


# Run the balanced GenEval comparison and show the saved baseline/head grids.
GENEVAL_SUMMARY = run_token_regret_geneval()
GENEVAL_GRID_OVERVIEW = show_geneval_saved_grids()
